# FlavourBench evidence and publication-readiness audit

**Purpose.** Reproduce the denominators behind the retrospective Season 0 paper, inspect endpoint uncertainty and judge survival, and keep the 2026-07-28 frontier refresh separate from the frozen pilot.

**Decision rule.** A technically complete provider run is not sufficient for a culinary leaderboard. Endpoint ranking additionally requires the declared comparison minimum, a usable comparison graph, qualified human criterion evidence, reviewed items, and a reproducible Epicure release.

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ANALYSIS = ROOT / (
    "artifacts/season0/analysis-v6/"
    "season0-automated-analysis-ab45eff77098a97fc05ef7ee5ca689b00724381e4bd8c6f7e4dd60c86fb61d97.json"
)
MANIFEST = ROOT / (
    "artifacts/season0/manifests/"
    "season0-model-manifest-3919def66686b4bd939c94cdd89659f63ae2afbbf03288413129e2ea8d6b83d2.json"
)
PAPER_DATA = ROOT.parent / "paper/flavourbench/generated/pilot"

analysis = json.loads(ANALYSIS.read_text())
manifest = json.loads(MANIFEST.read_text())
print({"analysis": ANALYSIS.name, "manifest": MANIFEST.name})

## 1. Integrity and fixed denominators

The analysis artifact is content addressed. The check below removes the self-referential digest field and reproduces its canonical SHA-256.

In [ ]:
def canonical_sha256(document):
    payload = dict(document)
    expected = payload.pop("artifact_sha256")
    rendered = json.dumps(
        payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    ).encode()
    return expected, hashlib.sha256(rendered).hexdigest()


expected, observed = canonical_sha256(analysis)
assert observed == expected == "ab45eff77098a97fc05ef7ee5ca689b00724381e4bd8c6f7e4dd60c86fb61d97"
counts = analysis["counts"]
integrity = pd.Series(
    {
        "attempted target arms": counts["scored_arms"],
        "recorded Epicure calls": counts["recorded_epicure_calls_including_partial"],
        "planned comparisons": counts["comparison_manifest_rows"],
        "judgment records": counts["judgment_records"],
        "primary consensuses": counts["consensus_available"],
    }
)
integrity

## 2. Endpoint comparison uncertainty

Every endpoint falls below the manuscript's 100-comparison reporting threshold. The table shows the task-bootstrap ridge diagnostic, not a validated culinary-quality ranking.

In [ ]:
model = pd.read_csv(PAPER_DATA / "pilot-model-uncertainty.csv")
model["interval_width"] = model["bootstrap_upper"] - model["bootstrap_lower"]
model["below_threshold"] = model["n"] < 100
assert model["below_threshold"].all()
assert model["complete_separation"].sum() == 1
model[
    [
        "model",
        "n",
        "wins",
        "ties",
        "losses",
        "bootstrap_median",
        "bootstrap_lower",
        "bootstrap_upper",
        "highest_resample_fraction",
        "complete_separation",
    ]
].sort_values("bootstrap_median", ascending=False)

## 3. Judge survival and paired estimand sensitivity

Orientation replication is a real quality control, but here it creates strong, model-dependent missingness. The paired result also changes with weighting and admission.

In [ ]:
judge = pd.read_csv(PAPER_DATA / "pilot-measurement-integrity.csv")
judge = judge[judge["panel"] == "judge"].pivot(index="label", columns="metric", values="rate")
uplift = pd.read_csv(PAPER_DATA / "pilot-epicure-robustness.csv")
reliability = pd.read_csv(PAPER_DATA / "pilot-condition-reliability.csv").iloc[0]
display(judge.sort_values("eligible_vote_yield", ascending=False))
display(uplift[["label", "estimate", "lower", "upper", "wins", "ties", "losses", "n"]])
print(
    {
        "on_minus_off_success_risk": reliability["realized_success_proportion_difference"],
        "interval": [
            reliability["lower_95_task_bootstrap"],
            reliability["upper_95_task_bootstrap"],
        ],
    }
)

## 4. Frontier-refresh availability evidence

The refresh is append-only and unranked. Failed infrastructure or entitlement checks are not converted into model-quality losses.

In [ ]:
summaries = sorted(
    (ROOT / "artifacts/frontier-refresh/2026-07-28").glob(
        "compatibility*/frontier-refresh-contract-summary-*.json"
    )
)
refresh_rows = []
for path in summaries:
    document = json.loads(path.read_text())
    for arm in document["artifacts"]:
        refresh_rows.append(
            {
                "batch": path.parent.name,
                "provider": arm["provider"],
                "model": arm["display_name"],
                "status": arm["status"],
                "provider_calls": arm["provider_calls"],
                "epicure_calls": arm["epicure_calls"],
                "error_type": arm.get("error_type"),
            }
        )
refresh = pd.DataFrame(refresh_rows)
refresh

## 5. Publication decision

The retrospective evidence is suitable for a measurement-audit paper, not a quality leaderboard. Four blockers are decisive:

1. no endpoint reaches 100 valid automated-consensus comparisons;
2. one endpoint is completely separated and the ordering is unstable under task resampling;
3. no qualified independent human criterion cohort or completed item review exists; and
4. the executed Epicure bundle is content addressed but lacks a redistributable, independently reproducible lineage.

The newest-model refresh cannot repair those issues retroactively. Opus 5 and Sonnet 5 were discovered as exact Bedrock global profiles, but the account denied inference entitlement. Kimi K3 and GLM 5.2 were frozen to exact OpenRouter endpoints, but the provider account did not admit the contract-smoke batch. Those are route-availability observations only.